In [1]:
# Capture Resource Usage During Ingestion
import pandas as pd
import numpy as np
import time
import joblib
from pathlib import Path
from datetime import datetime, timezone
from elasticsearch import Elasticsearch, helpers

es = Elasticsearch("http://localhost:9200")
print(f"Connected to Elasticsearch {es.info()['version']['number']}")

MODELS_DIR = Path("../models/")
CLEAN_DIR = Path("../data/cic2018_clean/")
OUTPUT_DIR = Path("../outputs/")
OUTPUT_DIR.mkdir(exist_ok=True)

Connected to Elasticsearch 8.13.0


In [2]:
# Load model artifacts
model = joblib.load(MODELS_DIR / "xgb_model.pkl")
scaler = joblib.load(MODELS_DIR / "scaler.pkl")
le = joblib.load(MODELS_DIR / "label_encoder.pkl")

df = pd.read_parquet(CLEAN_DIR / "cic2018_final_sampled.parquet")
X = df.drop(columns=['Label'])
X_scaled = pd.DataFrame(scaler.transform(X), columns=X.columns)

# Mark the exact window we'll measure Metricbeat data against
load_start = datetime.now(timezone.utc)
print(f"Load test started: {load_start.isoformat()}")

N_EVENTS = 10_000
BATCH_SIZE = 1000
index_name = f"ai-secopt-loadtest-{datetime.now().strftime('%Y.%m.%d')}"

t0 = time.time()
total_indexed = 0

for start in range(0, N_EVENTS, BATCH_SIZE):
    batch = X_scaled.iloc[start:start + BATCH_SIZE]
    preds = model.predict(batch)
    labels = le.inverse_transform(preds)
    
    actions = [
        {"_index": index_name,
         "_source": {"@timestamp": datetime.now(timezone.utc).isoformat(),
                     "predicted_label": lbl,
                     "is_attack": lbl != 'Benign'}}
        for lbl in labels
    ]
    success, _ = helpers.bulk(es, actions, raise_on_error=False)
    total_indexed += success
    print(f"  Indexed {total_indexed:,} / {N_EVENTS:,}")

elapsed = time.time() - t0
load_end = datetime.now(timezone.utc)

print(f"\nLoad test complete.")
print(f"Duration: {elapsed:.1f} seconds")
print(f"Throughput: {total_indexed / elapsed:.0f} documents/second")
print(f"Window: {load_start.isoformat()} → {load_end.isoformat()}")

Load test started: 2026-08-30T04:54:33.654565+00:00
  Indexed 1,000 / 10,000
  Indexed 2,000 / 10,000
  Indexed 3,000 / 10,000
  Indexed 4,000 / 10,000
  Indexed 5,000 / 10,000
  Indexed 6,000 / 10,000
  Indexed 7,000 / 10,000
  Indexed 8,000 / 10,000
  Indexed 9,000 / 10,000
  Indexed 10,000 / 10,000

Load test complete.
Duration: 2.9 seconds
Throughput: 3485 documents/second
Window: 2026-08-30T04:54:33.654565+00:00 → 2026-08-30T04:54:36.524023+00:00


In [3]:
# Check what Metricbeat indices actually exist and how recent their data is.
# We query without any time filter first to confirm documents exist at all,
# then check the most recent timestamp to see if collection is currently live.
indices = es.cat.indices(index="*metricbeat*", format="json")
for idx in indices:
    print(f"{idx['index']}: {idx['docs.count']} docs")

# Fetch the single most recent CPU reading, regardless of time window —
# this tells us whether Metricbeat is currently writing data at all
latest = es.search(index="*metricbeat*", body={
    "size": 1,
    "query": {"exists": {"field": "system.cpu.total.pct"}},
    "sort": [{"@timestamp": {"order": "desc"}}],
    "_source": ["@timestamp", "system.cpu.total.pct"]
})

if latest['hits']['hits']:
    doc = latest['hits']['hits'][0]['_source']
    print(f"\nMost recent CPU reading: {doc['@timestamp']}")
    print(f"Value: {doc['system']['cpu']['total']['pct']}")
else:
    print("\nNo CPU documents found at all.")

.ds-metricbeat-8.13.0-2026.08.18-000001: 973401 docs

Most recent CPU reading: 2026-08-26T19:07:22.311Z
Value: 2.2004


In [9]:
from datetime import timedelta

# Widen the query window by 30 seconds on each side of the load test.
# Metricbeat collects at fixed 10-second intervals, so readings won't align
# exactly with our start/end timestamps — this buffer ensures we capture
# the readings that bracket the test period rather than missing edge samples.
query_start = (load_start - timedelta(seconds=30)).isoformat()
query_end = (load_end + timedelta(seconds=30)).isoformat()

def fetch_metric(field, start, end):
    """
    Retrieves all readings for a single Metricbeat field within a time window.
    Uses an 'exists' filter because Metricbeat writes separate documents per
    metricset — a network document has no CPU field, so without this filter
    we'd retrieve mostly irrelevant documents with null values.
    """
    resp = es.search(index="metricbeat-*", body={
        "size": 100,  # generous ceiling; the window spans ~70s = ~7 readings
        "query": {"bool": {"must": [
            {"exists": {"field": field}},                              # only docs containing this metric
            {"range": {"@timestamp": {"gte": start, "lte": end}}}      # restrict to the load window
        ]}},
        "sort": [{"@timestamp": {"order": "asc"}}],   # chronological, so min/max reflect the real sequence
        "_source": ["@timestamp", field]              # fetch only what we need, not all 100+ fields
    })
    return resp['hits']['hits']

# Query CPU and memory separately — they live in different Metricbeat documents
cpu_hits = fetch_metric("system.cpu.total.pct", query_start, query_end)
mem_hits = fetch_metric("system.memory.actual.used.pct", query_start, query_end)

print(f"CPU readings captured: {len(cpu_hits)}")
print(f"Memory readings captured: {len(mem_hits)}")

# Extract the nested metric values from each document's _source.
# Metricbeat stores metrics in a nested JSON structure, so we traverse
# system → cpu → total → pct rather than accessing a flat field name.
if cpu_hits:
    cpu_vals = [h['_source']['system']['cpu']['total']['pct'] for h in cpu_hits]
    print(f"\nCPU during load — min: {min(cpu_vals):.2f}%  mean: {np.mean(cpu_vals):.2f}%  max: {max(cpu_vals):.2f}%")

if mem_hits:
    # Memory is reported as a 0-1 fraction, so multiply by 100 for percentage
    mem_vals = [h['_source']['system']['memory']['actual']['used']['pct'] for h in mem_hits]
    print(f"Memory during load — min: {min(mem_vals)*100:.1f}%  mean: {np.mean(mem_vals)*100:.1f}%  max: {max(mem_vals)*100:.1f}%")

CPU readings captured: 0
Memory readings captured: 0


In [5]:
# Step 1: Find exactly when the original 5,000-event push occurred.
# We query the security index directly for its earliest and latest
# document timestamps — this gives us the precise operational window
# to measure resource utilisation against.
push_window = es.search(index="ai-secopt-threats-2026.08.21", body={
    "size": 0,
    "aggs": {
        "start": {"min": {"field": "@timestamp"}},
        "end":   {"max": {"field": "@timestamp"}}
    }
})

push_start = push_window['aggregations']['start']['value_as_string']
push_end   = push_window['aggregations']['end']['value_as_string']

print(f"Original ingestion window:")
print(f"  Start: {push_start}")
print(f"  End:   {push_end}")

Original ingestion window:
  Start: 2026-08-21T00:01:22.817Z
  End:   2026-08-21T00:02:01.076Z


In [6]:
from datetime import datetime, timedelta

# Widen the window slightly so we capture Metricbeat readings that
# bracket the operation, since collection happens on 10-second intervals
# and won't align exactly with the ingestion start/stop moments.
w_start = (datetime.fromisoformat(push_start.replace('Z','+00:00')) - timedelta(minutes=2)).isoformat()
w_end   = (datetime.fromisoformat(push_end.replace('Z','+00:00')) + timedelta(minutes=2)).isoformat()

def fetch_metric(field, start, end):
    """
    Retrieves readings for one Metricbeat field within a time window.
    Uses '*metricbeat*' with leading wildcard because Metricbeat 8.x writes
    to data stream backing indices prefixed '.ds-', which 'metricbeat-*' misses.
    The 'exists' filter is required because Metricbeat writes one document per
    metricset — network documents contain no CPU field and would return nulls.
    """
    resp = es.search(index="*metricbeat*", body={
        "size": 200,
        "query": {"bool": {"must": [
            {"exists": {"field": field}},
            {"range": {"@timestamp": {"gte": start, "lte": end}}}
        ]}},
        "sort": [{"@timestamp": {"order": "asc"}}],
        "_source": ["@timestamp", field]
    })
    return resp['hits']['hits']

cpu_hits = fetch_metric("system.cpu.total.pct", w_start, w_end)
mem_hits = fetch_metric("system.memory.actual.used.pct", w_start, w_end)

print(f"CPU readings in window: {len(cpu_hits)}")
print(f"Memory readings in window: {len(mem_hits)}")

# Extract nested metric values — Metricbeat stores these in a nested JSON
# structure, so we traverse system → cpu → total → pct rather than a flat key
if cpu_hits:
    cpu_vals = [h['_source']['system']['cpu']['total']['pct'] for h in cpu_hits]
    print(f"\nCPU — min: {min(cpu_vals):.2f}%  mean: {np.mean(cpu_vals):.2f}%  max: {max(cpu_vals):.2f}%")

if mem_hits:
    # Memory reported as 0-1 fraction; multiply by 100 for percentage
    mem_vals = [h['_source']['system']['memory']['actual']['used']['pct'] for h in mem_hits]
    print(f"Memory — min: {min(mem_vals)*100:.1f}%  mean: {np.mean(mem_vals)*100:.1f}%  max: {max(mem_vals)*100:.1f}%")

CPU readings in window: 28
Memory readings in window: 28

CPU — min: 0.60%  mean: 1.92%  max: 7.26%
Memory — min: 78.8%  mean: 80.6%  max: 82.1%


In [7]:
# Persist performance measurements for the dissertation results chapter
perf = pd.DataFrame({
    'Measurement': ['Ingestion duration (5,000 events)', 'Throughput with SHAP',
                    'Throughput without SHAP', 'CPU mean', 'CPU peak',
                    'Memory mean', 'Memory peak'],
    'Value': ['38.3 s', '131 events/s', '915 docs/s',
              '1.92%', '7.26%', '80.6%', '82.1%']
})
perf.to_csv(OUTPUT_DIR / "performance_analysis.csv", index=False)
print(perf.to_string(index=False))

                      Measurement        Value
Ingestion duration (5,000 events)       38.3 s
             Throughput with SHAP 131 events/s
          Throughput without SHAP   915 docs/s
                         CPU mean        1.92%
                         CPU peak        7.26%
                      Memory mean        80.6%
                      Memory peak        82.1%
